# مُرشِد — تشغيل نظام ترشيح المطاعم الذكي على Google Colab

شغّلي الخلايا بالترتيب من فوق لتحت (زر ▶ على يسار كل خلية).

**قبل البداية:** حطي مفتاح OpenAI في Secrets حق كولاب: اضغطي أيقونة المفتاح 🔑 في الشريط الجانبي اليسار ← **Add new secret** ← الاسم `OPENAI_API_KEY` والقيمة مفتاحك ← فعّلي **Notebook access**.

إذا ما أضفتيه هناك، الخلية 3 بتطلبه منك.

## 1) تنزيل المشروع من GitHub

In [ ]:
!rm -rf smart-restaurant-recommender
!git clone -q https://github.com/yarasaeed10/smart-restaurant-recommender.git
%cd smart-restaurant-recommender
!ls

## 2) تثبيت المكتبات (تاخذ دقيقة أو دقيقتين)

In [ ]:
!grep -v pysqlite3 requirements.txt > requirements_colab.txt
!pip install -q -r requirements_colab.txt
print('تم التثبيت ✓')

## 3) إضافة مفتاح OpenAI بشكل آمن
المفتاح ينحفظ في ملف `.env` داخل جلسة كولاب فقط، وما ينرفع على GitHub.

In [ ]:
import os
from getpass import getpass
try:
    from google.colab import userdata
    key = userdata.get('OPENAI_API_KEY')
except Exception:
    key = None
if not key:
    key = getpass('الصقي مفتاح OpenAI هنا (يبدأ بـ sk-): ')
with open('.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={key.strip()}\nOPENAI_MODEL=gpt-4o-mini\n')
os.environ['OPENAI_API_KEY'] = key.strip()
print('تم حفظ المفتاح ✓' if key.strip().startswith('sk-') else '⚠️ المفتاح لا يبدو صحيحًا — النظام بيشتغل بالبحث المحلي')

## 4) تشغيل الاختبارات الآلية

In [ ]:
!python -m pytest -q

## 5) تجربة سريعة بدون واجهة
هذي الخلية تشغّل سلسلة LangChain كاملة وتطبع الترشيحات. غيّري الطلب وجربي.

In [ ]:
from src.config import get_settings
from src.data_loader import load_restaurants
from src.database import Database
from src.chain import RecommendationEngine

settings = get_settings()
df = load_restaurants(settings.data_path)
db = Database(settings.db_path)
engine = RecommendationEngine(settings, df, db)
print('OpenAI متصل ✓' if engine.llm_ready else f'وضع البحث المحلي: {engine.init_error.user_message if engine.init_error else ""}')

session = db.create_session()
for q in ['أبي مطعم بحري في جدة مناسب للعائلة وسعره متوسط', 'طيب أبي أرخص']:
    r = engine.recommend(q, session)
    print('\n🧑', q)
    print('فهمت طلبك:', r.preferences)
    if r.warning: print('⚠️', r.warning)
    print('🍽️', r.summary)
    for i, rec in enumerate(r.recommendations, 1):
        print(f"  {i}. {rec['name']} ({rec['match_score']}%) — {rec['reason']}")

## 6) تشغيل الواجهة والحصول على رابط
الخلية تشغّل Streamlit وتعطيك رابط ينتهي بـ `trycloudflare.com`. افتحيه في تبويب جديد.

خلي الخلية شغالة طول ما أنتِ تستخدمين التطبيق. إذا وقفتيها أو انقطعت جلسة كولاب، يقف الرابط.

In [ ]:
import subprocess, time, re, os
if not os.path.exists('cloudflared'):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                 stdout=open('streamlit.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\n✅ رابط التطبيق:', m.group(0))
        print('افتحيه في تبويب جديد (ممكن ياخذ 10–20 ثانية أول مرة)')
        break

### إذا ما فتح الرابط
شغّلي هذي الخلية وشوفي آخر سطور سجل Streamlit:

In [ ]:
!tail -20 streamlit.log